# Explorar las conexiones de datos con correlaciones II 📊🔗 

## Objetivos 🎯

1. Integrar múltiples técnicas de correlación en un solo análisis: Pearson/Spearman, point-biserial, Cramer’s V.
2. Identificar relaciones significativas entre variables mediante el uso de scatterplots y heatmaps.
3. Detectar correlaciones engañosas
4. Documentar supuestos y limitaciones
5. Convertir hallazgos en recomendaciones de negocio
6. Escribir un reporte profesional

## Analisis de CSNAT 2025 📘🎓

En este estudio se analizan los resultados de la **CSNAT (College Scholastic and Numerical Aptitude Test)**, una prueba estandarizada de **aptitud escolar y razonamiento académico** aplicada a estudiantes que aspiran a ingresar a la universidad. La métrica **csnat** se expresa en una escala de **700 a 1300 puntos** y busca capturar de manera integral habilidades académicas previas al ingreso a la educación superior.

Los datos corresponden a la **aplicación 2025 en la región oeste del país** y reúnen información demográfica, académica y contextual de los estudiantes evaluados. Además del puntaje csnat, la base incluye variables como edad, antecedentes educativos familiares, hábitos de estudio, desempeño académico histórico, actividades extracurriculares y condiciones de contexto personal.


El interés principal de este análisis es **explorar qué variables se asocian con el puntaje csnat y en qué magnitud**, utilizando un enfoque de correlación adecuado al tipo de datos. Dado que el puntaje pretende reflejar aptitud académica general, es relevante entender si existen **patrones consistentes** entre csnat y factores académicos, conductuales o contextuales, sin asumir relaciones causales.



### ❓ Preguntas de interés del análisis

A partir de este contexto, surgen preguntas como:
- ¿Qué tan asociadas están las **notas históricas** con el puntaje csnat?
- ¿Existen diferencias en el puntaje según **hábitos de estudio o estilo de vida**?
- ¿Algunas variables contextuales muestran asociaciones negativas con el desempeño?
- ¿Las asociaciones observadas son consistentes o cambian al analizar por segmentos?
- ¿Qué relaciones podrían reflejar **desigualdades estructurales** más que diferencias reales de aptitud?



### 🧠 Contexto y alcance de los datos

Es importante destacar que los datos:
- Corresponden a **una sola aplicación (2025)** y a **una región específica**
- Reflejan condiciones y decisiones previas, no características intrínsecas de los estudiantes
- Incluyen variables que pueden funcionar como **proxies socioeconómicos**

Por ello, los resultados deben interpretarse con cautela, evitando conclusiones deterministas o usos indebidos de las asociaciones encontradas.




### Carga de herramientas y datos

In [35]:
import pandas as pd
import numpy as np
from scipy.stats import pointbiserialr
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import  OrdinalEncoder, LabelEncoder

In [36]:
def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum().sum()
    r, k = contingency.shape
    return np.round(np.sqrt(chi2 / (n * (min(r, k) - 1))),2)

In [37]:
def corr_agg(group,var1,var2,method='pearson'):
    return group[var1].corr(group[var2],method=method)

In [38]:
csnat=pd.read_csv('https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/08-da-v8-analizar-correlaciones/csnat_2025_west.csv')

In [ ]:
csnat.head()

### Limpieza y analisis exploratorio de los datos

- Describe las variables
- Evalua la calidad de los datos (consistencia, duplicados, atípicos)
- Explora con estadísticas y gráficos que consideres de interés. 


In [41]:
# Codificacion de variables ordinales

edu_level=['sin_estudios', 'primaria', 'secundaria', 'bachillerato', 'universidad', 'posgrado']
smoke_level=['no_fuma', 'fuma_poco', 'fumador_ocasional', 'fumador_constante']
hour_level=['<5', '<10', '<15', '15+']
ordinal_col={'father_education':edu_level, 'mother_education':edu_level,'smoking_frequency':smoke_level,'study_hours_extra':hour_level}
for col,levels in ordinal_col.items():
    oe = OrdinalEncoder(categories=[levels])
    csnat[col+"_ord"] =  oe.fit_transform(csnat[[col]])

# Codificación de dicotómicas
categorical_col= ['school_type','extracurricular']
for col in categorical_col:
    le = LabelEncoder()
    csnat[col+"_lab"] =  le.fit_transform(csnat[col])


csnat.head()

,age,father_education,mother_education,school_type,scholarship,smoking_frequency,study_hours_extra,extracurricular,work_hours,math_grade,language_grade,aptitude_score,father_education_ord,mother_education_ord,smoking_frequency_ord,study_hours_extra_ord,school_type_lab,extracurricular_lab
0,20,posgrado,universidad,publico,0,no_fuma,<5,social,4.3,66.2,73.5,1300.0,5.0,4.0,0.0,0.0,1,2
1,21,bachillerato,primaria,publico,0,fumador_ocasional,15+,deportiva,3.3,81.6,64.3,1300.0,3.0,1.0,2.0,3.0,1,1
2,19,bachillerato,primaria,privado,0,no_fuma,15+,deportiva,19.2,71.1,86.3,1276.0,3.0,1.0,0.0,3.0,0,1
3,21,posgrado,sin_estudios,publico,0,fuma_poco,<5,cultural,16.5,54.3,74.5,1194.0,5.0,0.0,1.0,0.0,1,0
4,21,universidad,posgrado,privado,0,no_fuma,<10,deportiva,10.4,86.7,79.6,1300.0,4.0,5.0,0.0,1.0,0,1


### Analisis de correlación 

Analiza la relación entre las variables con el puntaje *csnat*.

- Para las variables numéricas:
    - Crea un `pairplot` con el conjunto de variables
    - Calcula la matriz de correlación
    - Muestra el heatmap asociado
    - Discute la correlación con la variable de interes
    - Discute los siguientes supuestos
        - Describe las variables que tienen correlación positiva y negativa
        - ¿Hay evidencia que el grado de estudios de los padres está co-relacionado con el desempeño de los hijos? Discute
        - Opina sobre la relación encontrada con el tabaquismo y lo delicado sobre cómo comunicar el hallazgo

- Para variables categoricas, dicotómicas:
    - Para cada variable
        - Agrupa y muestra el puntaje promedio `csnat` por categoria
        - Haz un boxplot del `csnat` separando por los niveles de la variable
        - Calcula la correlación con `csnat` para las dicotómicas
    - Discute los siguientes supuestos:
        - Los estudiantes con beca tienden a tener mejor puntaje que los que no
        - Las actividades extracurriculares impactan en el puntaje
        - Los estudiantes de escuelas privadas tienen mejor puntaje en la prueba



## 🚀 Para seguir aprendiendo :


- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨